In [ ]:
MAPPING FILE: mplt_CDM_ROW_WID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_ROW_WID
Migrated from IICS mapping: mplt_CDM_ROW_WID

Purpose:
This pipeline retrieves the maximum ROW_WID and associated TABLE_NAME from a target table using a lookup transformation,
calculates a new ROW_WID based on conditional logic, and outputs the result to a target table in Snowflake Cloud Data Warehouse.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMRowWidPipeline:
    """
    ETL Pipeline for calculating ROW_WID.

    Sources: Custom Table
    Targets: Snowflake Cloud Data Warehouse Table
    Transformation Logic: Lookup maximum ROW_WID, calculate new ROW_WID, and output to target.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        Optimizations:
        - Predicate pushdown for filters.
        - Column pruning to select only required fields.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: Custom Table")
        logger.info("Data extraction complete")
        return df

    def lookup_max_row_wid(self, table_name: str) -> DataFrame:
        """
        Perform lookup to retrieve maximum ROW_WID and associated TABLE_NAME.

        Args:
            table_name: Name of the table to perform lookup on.

        Returns:
            DataFrame with ROW_WID and TABLE_NAME.
        """
        logger.info("Performing lookup for maximum ROW_WID")

        lookup_query = f"""
            SELECT NVL(MAX(ROW_WID), 0) AS ROW_WID, TABLE_NAME
            FROM {self.config.get('lookup_table')}
            WHERE TABLE_NAME = '{table_name}'
        """
        lookup_df = self.spark.sql(lookup_query)
        log_df_info(lookup_df, "Lookup: Maximum ROW_WID")
        return lookup_df

    def transform(self, source_df: DataFrame, lookup_df: DataFrame) -> DataFrame:
        """
        Apply transformations to calculate new ROW_WID.

        Args:
            source_df: Source DataFrame.
            lookup_df: Lookup DataFrame.

        Returns:
            Transformed DataFrame with calculated ROW_WID.
        """
        logger.info("Applying transformations")

        # Join source with lookup results
        df = source_df.join(
            F.broadcast(lookup_df),
            source_df["TABLE_NAME"] == lookup_df["TABLE_NAME"],
            "left"
        )

        # Calculate new ROW_WID
        df = df.withColumn(
            "ROW_WID",
            F.when(F.col("ROW_WID").isNull(), 0).otherwise(F.col("ROW_WID")) + 1
        )

        log_df_info(df, "After transformations")
        return df

    def load(self, df: DataFrame):
        """
        Load data to target.

        Optimizations:
        - Delta Lake for ACID compliance.
        - Partitioning for query performance.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', 'path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .partitionBy(*self.config.get('partition_columns', []))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        # Optimize target table
        if self.config.get('optimize_target', True):
            logger.info("Optimizing target table")
            zorder_cols = self.config.get('zorder_columns', [])
            if zorder_cols:
                self.spark.sql(f"""
                    OPTIMIZE delta.`{target_path}`
                    ZORDER BY ({', '.join(zorder_cols)})
                """)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Lookup
            lookup_df = self.lookup_max_row_wid(self.config.get('lookup_table_name'))

            # Transform
            transformed_df = self.transform(source_df, lookup_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'catalog.database.source_table',
    'source_local_path': r'path/to/source.csv',

    # Lookup configuration
    'lookup_table': 'catalog.database.lookup_table',
    'lookup_table_name': 'target_table_name',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',
    'partition_columns': ['TABLE_NAME'],
    'zorder_columns': ['ROW_WID'],

    # Performance configuration
    'shuffle_partitions': 200,
    'optimize_target': True
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_ROW_WID_Pipeline").getOrCreate()
    pipeline = CDMRowWidPipeline(spark, config)
    pipeline.execute()
MAPPING FILE: mplt_CDM_BATCH_ID.txt
====================================================================================================

"""
ETL Pipeline: mplt_CDM_BATCH_ID
Migrated from IICS mapping: mplt_CDM_BATCH_ID
Description: Processes batch IDs by checking for null values, performing a lookup to retrieve the maximum batch ID for a given source name, and outputs either the lookup value or a default value (-999) if the batch ID is null.
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table" (e.g., "CDM.CDM_BATCH_CTRLID").
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ

    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMBatchIDMapplet:
    """
    ETL Pipeline for processing batch IDs.

    Sources: CDM.CDM_BATCH_CTRLID
    Targets: Output Table
    Transformation Logic: Check for null values, perform lookup, and assign default values.
    """

    def __init__(self, spark: SparkSession, config: dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }

        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source.

        Optimizations:
        - Predicate pushdown for filters.
        - Column pruning to select only required fields.
        """
        logger.info("Starting data extraction")

        table_name = self.config.get('source_table', 'CDM.CDM_BATCH_CTRLID')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(
            table_name=table_name,
            local_file_path=local_file_path
        )

        log_df_info(df, "Source: CDM.CDM_BATCH_CTRLID")
        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        Transformation Logic:
        - Lookup maximum batch ID for the given source name.
        - Assign default value (-999) if batch ID is null.
        """
        logger.info("Applying transformations")

        # Perform lookup transformation
        lookup_table_name = self.config.get('lookup_table', 'CDM.CDM_BATCH_CTRLID')
        lookup_local_path = self.config.get('lookup_local_path', None)

        lookup_df = read_table(
            table_name=lookup_table_name,
            local_file_path=lookup_local_path
        )

        # Perform lookup join
        logger.info("Performing lookup transformation")
        lookup_df = lookup_df.groupBy("SOURCE_NAME").agg(
            F.max("BATCH_ID").alias("LKP_BATCH_ID")
        )

        df = df.join(
            F.broadcast(lookup_df),
            df["SOURCE_NAME"] == lookup_df["SOURCE_NAME"],
            "left"
        ).select(
            df["SOURCE_NAME"],
            lookup_df["LKP_BATCH_ID"]
        )

        # Null check and default value assignment
        logger.info("Applying null check and default value assignment")
        df = df.withColumn(
            "o_BATCH_ID",
            F.when(F.col("LKP_BATCH_ID").isNull(), F.lit(-999)).otherwise(F.col("LKP_BATCH_ID"))
        ).select(
            "SOURCE_NAME",
            "o_BATCH_ID"
        )

        log_df_info(df, "After transformations")
        return df

    def load(self, df: DataFrame):
        """
        Load data to target.

        Optimizations:
        - Delta Lake for ACID compliance.
        - Partitioning for query performance.
        """
        logger.info("Starting data load")

        target_path = self.config.get('target_path', '/path/to/target')

        log_df_info(df, "Before load")

        (df.write
            .format("delta")
            .mode(self.config.get('write_mode', 'overwrite'))
            .option("overwriteSchema", "true")
            .save(target_path)
        )

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")

        try:
            # Extract
            source_df = self.extract()

            # Transform
            transformed_df = self.transform(source_df)

            # Load
            self.load(transformed_df)

            logger.info("ETL pipeline completed successfully")

        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDM.CDM_BATCH_CTRLID',
    'source_local_path': r'path/to/CDM_BATCH_CTRLID.csv',

    # Target configuration
    'target_path': '/path/to/target',
    'write_mode': 'overwrite',

    # Lookup configuration
    'lookup_table': 'CDM.CDM_BATCH_CTRLID',
    'lookup_local_path': r'path/to/CDM_BATCH_CTRLID.csv',

    # Performance configuration
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("CDM_BATCH_ID_Mapplet").getOrCreate()
    pipeline = CDMBatchIDMapplet(spark, config)
    pipeline.execute()
MAPPING FILE: m_CDM_W_CLAIM_CD_SCD3_IU.txt
====================================================================================================

"""
ETL Pipeline: m_CDM_W_CLAIM_CD_SCD3_IU
Migrated from IICS mapping: m_CDM_W_CLAIM_CD_SCD3_IU
"""

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from typing import Dict
import logging
import os

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Utility functions
def read_table(table_name: str, local_file_path: str = None) -> DataFrame:
    """
    Read data from catalog (Databricks) or local file based on environment.

    Args:
        table_name: Full table name in format "catalog.database.table".
        local_file_path: Local CSV file path for non-Databricks environment.

    Returns:
        DataFrame from catalog or local file.
    """
    is_databricks = 'DATABRICKS_RUNTIME_VERSION' in os.environ
    if is_databricks:
        logger.info(f"Reading from catalog: {table_name}")
        return spark.table(table_name)
    else:
        if local_file_path is None:
            raise Exception(f"Local file path required for {table_name}")
        logger.info(f"Reading from local file: {local_file_path}")
        return spark.read.csv(local_file_path, header=True, inferSchema=True)

def log_df_info(df: DataFrame, step_name: str):
    """
    Log DataFrame information for debugging.

    Args:
        df: DataFrame to log.
        step_name: Description of the step.
    """
    try:
        logger.info(f"{step_name}: {len(df.columns)} columns")
        logger.info(f"Schema: {[f.name + ':' + str(f.dataType) for f in df.schema.fields]}")
    except Exception as e:
        logger.error(f"Error logging DataFrame info: {str(e)}")

class CDMClaimPipeline:
    """
    ETL Pipeline for processing SCD Type 3 logic for BUR field.

    Sources: CDH_GW_BUR
    Targets: W_CLAIM_CD_BUR_SCD3_I (Insert), W_CLAIM_CD_BUR_SCD3_U (Update)
    Transformation Logic: SCD Type 3 handling with lookup and router logic.
    """

    def __init__(self, spark: SparkSession, config: Dict):
        self.spark = spark
        self.config = config
        self._configure_spark()

    def _configure_spark(self):
        """
        Set Spark configurations for optimal performance.
        """
        configs = {
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.shuffle.partitions": str(self.config.get('shuffle_partitions', 200))
        }
        for key, value in configs.items():
            self.spark.conf.set(key, value)

    def extract(self) -> DataFrame:
        """
        Extract data from source table.

        IICS Source Qualifier equivalent.
        """
        logger.info("Starting data extraction")
        table_name = self.config.get('source_table', 'catalog.database.table')
        local_file_path = self.config.get('source_local_path', None)

        df = read_table(table_name=table_name, local_file_path=local_file_path)
        log_df_info(df, "Source: CDH_GW_BUR")

        # Apply SQL override logic
        df = df.select(
            F.col("POLICY_STATE"),
            F.col("BUR"),
            F.lit("GWCDH").alias("SOURCE_NAME")
        )
        logger.info("Data extraction complete")
        return df

    def transform(self, df: DataFrame) -> DataFrame:
        """
        Apply business transformations.

        IICS Transformation Logic equivalent.
        """
        logger.info("Applying transformations")

        # Step 2: EXP_BUR
        df = df.select(
            F.col("POLICY_STATE").alias("INTEGRATION_ID"),
            F.col("BUR"),
            F.col("SOURCE_NAME")
        )

        # Step 3: Lookup Transformation
        lookup_table_name = self.config.get('lookup_table', 'catalog.database.lookup_table')
        lookup_local_path = self.config.get('lookup_local_path', None)
        lookup_df = read_table(table_name=lookup_table_name, local_file_path=lookup_local_path)

        lookup_df = lookup_df.select("LKP_ROW_WID", "LKP_INTEGRATION_ID", "LKP_NEW_BUR")
        df = df.join(
            F.broadcast(lookup_df),
            df.INTEGRATION_ID == lookup_df.LKP_INTEGRATION_ID,
            "left"
        ).select(
            df["*"],
            lookup_df["LKP_ROW_WID"],
            lookup_df["LKP_NEW_BUR"]
        )

        # Step 4: EXP_Flag
        df = df.withColumn(
            "o_Flag",
            F.when(F.col("LKP_ROW_WID").isNull(), "I")
            .when(F.md5(F.col("BUR")) == F.md5(F.col("LKP_NEW_BUR")), "NC")
            .otherwise("U")
        ).withColumn("CDM_INSERT_DT", F.current_timestamp()) \
         .withColumn("CDM_UPDATE_DT", F.current_timestamp()) \
         .withColumn("TGT_TABLE_NAME", F.lit("W_CLAIM_CD_BUR_SCD3"))

        # Step 5: Router Transformation
        insert_df = df.filter(F.col("o_Flag") == "I").select("o_Flag", "CDM_INSERT_DT", "TGT_TABLE_NAME")
        update_df = df.filter(F.col("o_Flag") == "U").select("o_Flag", "CDM_UPDATE_DT", "TGT_TABLE_NAME")

        logger.info("Transformations complete")
        return insert_df, update_df

    def load(self, insert_df: DataFrame, update_df: DataFrame):
        """
        Load data into target tables.

        IICS Target equivalent.
        """
        logger.info("Starting data load")

        # Load Insert Data
        insert_target_path = self.config.get('insert_target_path', '/path/to/insert_target')
        log_df_info(insert_df, "Insert Data")
        insert_df.write.format("delta").mode("append").save(insert_target_path)

        # Load Update Data
        update_target_path = self.config.get('update_target_path', '/path/to/update_target')
        log_df_info(update_df, "Update Data")
        update_df.write.format("delta").mode("append").save(update_target_path)

        logger.info("Data load complete")

    def execute(self):
        """
        Execute the complete ETL pipeline.
        """
        logger.info(f"Starting ETL pipeline: {self.__class__.__name__}")
        try:
            # Extract
            source_df = self.extract()

            # Transform
            insert_df, update_df = self.transform(source_df)

            # Load
            self.load(insert_df, update_df)

            logger.info("ETL pipeline completed successfully")
        except Exception as e:
            logger.error(f"ETL pipeline failed: {str(e)}")
            raise


# Configuration
config = {
    # Source configuration
    'source_table': 'CDH_GW_BUR',
    'source_local_path': r'path/to/source.csv',

    # Target configuration
    'insert_target_path': '/path/to/insert_target',
    'update_target_path': '/path/to/update_target',

    # Lookup configuration
    'lookup_table': 'CDM.W_CLAIM_CD_BUR_SCD3',
    'lookup_local_path': r'path/to/lookup.csv',

    # Performance configuration
    'shuffle_partitions': 200
}

# Execute
if __name__ == "__main__":
    spark = SparkSession.builder.appName("ETL_Pipeline").getOrCreate()
    pipeline = CDMClaimPipeline(spark, config)
    pipeline.execute()